# Module 0 — Setup & RAG Architecture

**Duration:** ~30 minutes  
**Goal:** Verify your environment, understand the RAG concept, and see the full architecture before we build each piece.

---

## What is RAG?

**Retrieval-Augmented Generation (RAG)** combines a retrieval system with a language model:

```
  User Question
       │
       ▼
  ┌─────────────┐        ┌──────────────────┐
  │  Retriever  │──────▶│   Vector Store    │
  │ (semantic   │        │  (pre-indexed    │
  │  search)    │◀──────│   documents)     │
  └─────────────┘        └──────────────────┘
       │
       │  Top-k relevant chunks
       ▼
  ┌─────────────┐
  │   Prompt    │  = System prompt + Context chunks + User question
  │  Assembly   │
  └─────────────┘
       │
       ▼
  ┌─────────────┐
  │     LLM     │  Claude synthesises an answer grounded in the retrieved context
  │  (Claude)   │
  └─────────────┘
       │
       ▼
    Answer
```

### Why RAG instead of fine-tuning?

| | RAG | Fine-tuning |
|--|-----|-------------|
| Update knowledge | Add documents, re-index | Retrain model |
| Cost | Low (inference only) | High (GPU training) |
| Transparency | Can show sources | Black box |
| Hallucination risk | Reduced (grounded) | Higher |
| Setup time | Hours | Days/weeks |

---

## Workshop Tech Stack

```
openai               → LLM client (Netlight endpoint)
sentence-transformers → Embeddings (local, free)
chromadb             → Vector store (local, file-based)
ragas                → Evaluation framework
pdfplumber + pypdf   → PDF extraction for insurance docs
```

**No cloud lock-in.** Everything runs on your laptop.

---
## Step 1 — Check your environment

In [ ]:
import sys
print(f"Python {sys.version}")
assert sys.version_info >= (3, 11), "Please use Python 3.11+"
print("✓ Python version OK")

In [ ]:
# Check all required libraries
import importlib

required = [
    "openai",
    "sentence_transformers",
    "chromadb",
    "ragas",
    "pandas",
    "numpy",
    "matplotlib",
]

missing = []
for lib in required:
    try:
        importlib.import_module(lib)
        print(f"  ✓ {lib}")
    except ImportError:
        print(f"  ✗ {lib} — MISSING")
        missing.append(lib)

if missing:
    print(f"\nInstall missing: pip install {' '.join(missing)}")
else:
    print("\nAll libraries installed!")

In [ ]:
import os
from dotenv import load_dotenv

# Load .env file from project root
load_dotenv("../.env")

api_key = os.getenv("NETLIGHT_API_KEY", "")
if api_key and len(api_key) > 10:
    print(f"✓ NETLIGHT_API_KEY found ({api_key[:12]}...)")
else:
    print("✗ NETLIGHT_API_KEY not found or invalid")
    print("  Add your Netlight API key to .env as NETLIGHT_API_KEY")

In [ ]:
# Quick API test — calls Claude with a tiny prompt
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.ai/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)

response = client.chat.completions.create(
    model="claude-haiku-4-5",
    max_tokens=64,
    messages=[{"role": "user", "content": "Reply with exactly: RAG workshop ready!"}],
)
print(response.choices[0].message.content)

In [ ]:
# Quick embedding test — downloads the model on first run (~90 MB)
from sentence_transformers import SentenceTransformer

print("Loading embedding model (first run downloads ~90 MB)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

test_vec = embed_model.encode("Hello RAG workshop!")
print(f"✓ Embedding model loaded. Vector dimensions: {len(test_vec)}")

In [ ]:
# Check that the data was downloaded
from pathlib import Path
import json

corpus_path = Path("../data/raw/corpus.json")
if corpus_path.exists():
    with open(corpus_path) as f:
        corpus = json.load(f)
    total_chars = sum(len(a["content"]) for a in corpus)
    print(f"✓ Corpus loaded: {len(corpus)} articles, {total_chars:,} characters")
    print("  Topics:", [a["title"] for a in corpus[:5]], "...")
else:
    print("✗ Data not found. Run: python ../data/download_data.py")

---
## Step 2 — RAG Pipeline Preview

Before we build each piece, let's see the complete pipeline run end-to-end.
Don't worry about understanding every line — we'll cover each component in depth.

In [ ]:
# ─────────────────────────────────────────────────────────────
# PREVIEW: Full RAG pipeline in ~50 lines
# We'll build each step properly in the following modules.
# ─────────────────────────────────────────────────────────────

import json
import re
from pathlib import Path

from openai import OpenAI
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Load corpus
with open("../data/raw/corpus.json") as f:
    corpus = json.load(f)

# 2. Chunk documents (simple fixed-size split)
def simple_chunk(text: str, size: int = 500) -> list[str]:
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

chunks = []
for article in corpus:
    for chunk in simple_chunk(article["content"]):
        chunks.append({"text": chunk, "source": article["title"]})

print(f"Created {len(chunks)} chunks from {len(corpus)} articles")

# 3. Embed chunks and store in ChromaDB
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma = chromadb.Client()  # in-memory for the preview
collection = chroma.create_collection("preview")

texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True).tolist()
ids = [str(i) for i in range(len(chunks))]
metas = [{"source": c["source"]} for c in chunks]

collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metas)
print(f"Indexed {len(chunks)} chunks in ChromaDB")

In [ ]:
# 4. Retrieve + Generate
def rag_answer(question: str, top_k: int = 3) -> str:
    # Retrieve
    q_embedding = embed_model.encode(question).tolist()
    results = collection.query(query_embeddings=[q_embedding], n_results=top_k)
    context_chunks = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]

    # Build prompt
    context = "\n\n---\n\n".join(context_chunks)
    prompt = f"""Answer the question using ONLY the context below.
If the context does not contain enough information, say so.

Context:
{context}

Question: {question}
"""

    # Generate
    client = OpenAI(
        base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.ai/v1"),
        api_key=os.getenv("NETLIGHT_API_KEY"),
    )
    response = client.chat.completions.create(
        model="claude-haiku-4-5",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = response.choices[0].message.content

    return answer, context_chunks, sources


# Try it!
question = "What is reinsurance and why do insurance companies use it?"
answer, contexts, sources = rag_answer(question)

print(f"Question: {question}\n")
print(f"Answer:\n{answer}\n")
print(f"Sources: {list(set(sources))}")

---
## Key Concepts Glossary

| Term | Definition |
|------|------------|
| **Chunk** | A piece of text (usually 200–1000 tokens) extracted from a document |
| **Embedding** | A dense numerical vector representing the semantic meaning of text |
| **Vector Store** | A database optimised for similarity search on embeddings |
| **Cosine Similarity** | A metric (0–1) measuring how similar two vectors are in direction |
| **Top-k Retrieval** | Returning the k most similar chunks to the query |
| **Context Window** | The maximum text an LLM can process in a single call |
| **Hallucination** | When an LLM generates plausible-sounding but factually incorrect content |
| **Grounding** | Anchoring LLM output to retrieved facts to reduce hallucination |
| **Faithfulness** | A RAG quality metric: does the answer stay true to the retrieved context? |
| **Answer Relevancy** | A RAG quality metric: does the answer address the question? |

---

## Next: Module 1 — Chunking & Embeddings

Now we'll take the `simple_chunk` function above and understand:
- **Why chunking strategy matters** — too large vs too small
- **Different chunking approaches** — fixed-size, sentence, semantic
- **How embeddings work** — and how to choose the right model

Open `01_document_ingestion.ipynb` →